##**Step 1**: Install Spark

##**Step 2**: Setting up Spark

Before you can connect to a Spark cluster, Spark needs to be installed. The code below is boilerplate code that can be used to set-up Spark. Please note that this code will be leveraged in all the notebooks since each nodebook is a separate entity.

## **Step 3**. Import the lib

In [ ]:
# Colab-friendly setup for PySpark
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install -q "pyspark[connect]==3.5.1" "dataproc-spark-connect==0.8.3"

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("RDDPractice").getOrCreate()
sc = spark.sparkContext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.8 MB/s eta 0:00:00


## **Step 4**: Using DataFrames

Spark's core data structure is the Resilient Distributed Dataset (RDD), which is a low-level object designed to handle large datasets by distributing data across multiple nodes in a cluster. However, working directly with RDDs can be challenging. To simplify this, Spark provides DataFrames, which offer a higher-level abstraction built on top of RDDs.

A Spark DataFrame functions similarly to a SQL table, where columns represent attributes and rows represent observations. DataFrames not only make it easier to manipulate and analyze data, but they are also more optimized for complex operations compared to RDDs.

In [ ]:
## mount to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Read the Airlines.csv file from your Google Drive into a Spark DataFrame

In [ ]:
## Read RDD
df = spark.read.option("header", "true").csv("/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Dataset/airline.csv")
df.show(100)

+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|Year|Month|DayofMonth|DayOfWeek|DepTime|CRSDepTime|ArrTime|CRSArrTime|UniqueCarrier|FlightNum|TailNum|ActualElapsedTime|CRSElapsedTime|AirTime|ArrDelay|DepDelay|Origin|Dest|Distance|TaxiIn|TaxiOut|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2005|    1|         1|        6|   1211|      1216|   1451|      1502|           UA|      548

Output the column names and the "rows" and "column" counts

In [ ]:
#outputing the column names
df.columns #Column Names

['Year',
 'Month',
 'DayofMonth',
 'DayOfWeek',
 'DepTime',
 'CRSDepTime',
 'ArrTime',
 'CRSArrTime',
 'UniqueCarrier',
 'FlightNum',
 'TailNum',
 'ActualElapsedTime',
 'CRSElapsedTime',
 'AirTime',
 'ArrDelay',
 'DepDelay',
 'Origin',
 'Dest',
 'Distance',
 'TaxiIn',
 'TaxiOut',
 'Cancelled',
 'CancellationCode',
 'Diverted',
 'CarrierDelay',
 'WeatherDelay',
 'NASDelay',
 'SecurityDelay',
 'LateAircraftDelay']

In [ ]:
#Outputing the row count
df.count()  #Row Count
#Output the column count
len(df.columns) #Column Count

29

In [ ]:
df.dtypes

[('Year', 'string'),
 ('Month', 'string'),
 ('DayofMonth', 'string'),
 ('DayOfWeek', 'string'),
 ('DepTime', 'string'),
 ('CRSDepTime', 'string'),
 ('ArrTime', 'string'),
 ('CRSArrTime', 'string'),
 ('UniqueCarrier', 'string'),
 ('FlightNum', 'string'),
 ('TailNum', 'string'),
 ('ActualElapsedTime', 'string'),
 ('CRSElapsedTime', 'string'),
 ('AirTime', 'string'),
 ('ArrDelay', 'string'),
 ('DepDelay', 'string'),
 ('Origin', 'string'),
 ('Dest', 'string'),
 ('Distance', 'string'),
 ('TaxiIn', 'string'),
 ('TaxiOut', 'string'),
 ('Cancelled', 'string'),
 ('CancellationCode', 'string'),
 ('Diverted', 'string'),
 ('CarrierDelay', 'string'),
 ('WeatherDelay', 'string'),
 ('NASDelay', 'string'),
 ('SecurityDelay', 'string'),
 ('LateAircraftDelay', 'string')]

In order to examine the summary of any particular column of a DataFrame, we use the describe method.

The describe "method" gives us the statistical summary of the given column

In [ ]:
#Examine the summary of the "DepDelay" column
df.describe('DepDelay').show()

+-------+-----------------+
|summary|         DepDelay|
+-------+-----------------+
|  count|           539895|
|   mean|11.16137993946179|
| stddev|34.37881570185133|
|    min|               -1|
|    max|               NA|
+-------+-----------------+



In order to select particular columns from the DataFrame, use the select method

In [ ]:
df.select('ArrDelay','DepDelay').show()

+--------+--------+
|ArrDelay|DepDelay|
+--------+--------+
|     -11|      -5|
|     -15|      -7|
|      -8|      -3|
|      NA|      NA|
|       2|      -5|
|       1|      -1|
|      75|      75|
|     -17|      -2|
|     -22|      -9|
|      65|      83|
|     -16|      -4|
|     -22|      -7|
|     -23|      -5|
|     -14|       3|
|     -17|      -6|
|     -16|      -4|
|     -15|      -6|
|     -13|      -4|
|     -10|      -5|
|      -1|       2|
+--------+--------+
only showing top 20 rows



Selecting Distinct Multiple Columns

In [ ]:
df.select('ArrDelay','DepDelay').distinct().show()

+--------+--------+
|ArrDelay|DepDelay|
+--------+--------+
|     -23|      -5|
|     -20|      -6|
|       5|      10|
|      58|      62|
|      41|      31|
|      71|      77|
|      37|      45|
|       6|      -9|
|      65|      73|
|     102|      83|
|      56|      33|
|     123|     140|
|      80|      33|
|      55|      41|
|      69|      77|
|     111|     100|
|     701|     694|
|     101|     100|
|     136|     131|
|     112|      97|
+--------+--------+
only showing top 20 rows



## **Step 5.** Filtering Data

In [ ]:
df.filter(df.Origin=='SFO').show()

+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|Year|Month|DayofMonth|DayOfWeek|DepTime|CRSDepTime|ArrTime|CRSArrTime|UniqueCarrier|FlightNum|TailNum|ActualElapsedTime|CRSElapsedTime|AirTime|ArrDelay|DepDelay|Origin|Dest|Distance|TaxiIn|TaxiOut|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2005|    1|         1|        6|   1211|      1216|   1451|      1502|           UA|      548

Filtering Data (Multiple Parameters)

In [ ]:
df.filter((df.Origin=='SFO')&(df.Dest=='PDX')).show()

+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|Year|Month|DayofMonth|DayOfWeek|DepTime|CRSDepTime|ArrTime|CRSArrTime|UniqueCarrier|FlightNum|TailNum|ActualElapsedTime|CRSElapsedTime|AirTime|ArrDelay|DepDelay|Origin|Dest|Distance|TaxiIn|TaxiOut|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2005|    1|         6|        4|   1830|      1831|   2005|      2012|           UA|      558

## **Step 6**: Performing SQL Queries
- SQL queries can be passes directly to any DataFrame; for that, we need to create a table from the DataFrame using the registerTempTable

In [ ]:
# old (deprecated)
# df.registerTempTable('airlines')

# new (Spark 2.0+)
df.createOrReplaceTempView("airlines")

Typically the entry point into all SQL functionality in Spark is the SQLContext class. To create a basic instance of this call, all we need is a SparkContext reference.

In [ ]:
from pyspark.sql import SQLContext
sqlContext = SQLContext(spark)
sqlContext

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
sqlContext.sql('select * from airlines').show()

+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|Year|Month|DayofMonth|DayOfWeek|DepTime|CRSDepTime|ArrTime|CRSArrTime|UniqueCarrier|FlightNum|TailNum|ActualElapsedTime|CRSElapsedTime|AirTime|ArrDelay|DepDelay|Origin|Dest|Distance|TaxiIn|TaxiOut|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2005|    1|         1|        6|   1211|      1216|   1451|      1502|           UA|      548

In [ ]:
sqlContext.sql('select distinct(Dest) from airlines').show(20)

+----+
|Dest|
+----+
| MSY|
| SNA|
| BUR|
| IDA|
| EUG|
| RDM|
| MOD|
| CEC|
| LIH|
| IAH|
| HNL|
| CVG|
| SJC|
| RDD|
| AUS|
| GJT|
| BFL|
| RNO|
| BOS|
| EWR|
+----+
only showing top 20 rows



In [ ]:
print(df.count())

539895


In [ ]:
print(df.take(3))

[Row(Year='2005', Month='1', DayofMonth='1', DayOfWeek='6', DepTime='1211', CRSDepTime='1216', ArrTime='1451', CRSArrTime='1502', UniqueCarrier='UA', FlightNum='548', TailNum='N341UA', ActualElapsedTime='100', CRSElapsedTime='106', AirTime='81', ArrDelay='-11', DepDelay='-5', Origin='SFO', Dest='SLC', Distance='599', TaxiIn='2', TaxiOut='17', Cancelled='0', CancellationCode=None, Diverted='0', CarrierDelay='0', WeatherDelay='0', NASDelay='0', SecurityDelay='0', LateAircraftDelay='0'), Row(Year='2005', Month='1', DayofMonth='2', DayOfWeek='7', DepTime='1209', CRSDepTime='1216', ArrTime='1447', CRSArrTime='1502', UniqueCarrier='UA', FlightNum='548', TailNum='N398UA', ActualElapsedTime='98', CRSElapsedTime='106', AirTime='79', ArrDelay='-15', DepDelay='-7', Origin='SFO', Dest='SLC', Distance='599', TaxiIn='2', TaxiOut='17', Cancelled='0', CancellationCode=None, Diverted='0', CarrierDelay='0', WeatherDelay='0', NASDelay='0', SecurityDelay='0', LateAircraftDelay='0'), Row(Year='2005', Month

In [ ]:
## Filter#3
spark.sql("""SELECT DISTINCT Origin, Dest, Distance
FROM airlines WHERE distance > 1000
ORDER BY distance DESC""").show(10)

+------+----+--------+
|Origin|Dest|Distance|
+------+----+--------+
|   SFO| BOS|    2704|
|   SFO| JFK|    2586|
|   SFO| MIA|    2585|
|   SFO| EWR|    2565|
|   SFO| PHL|    2521|
|   SFO| BWI|    2457|
|   SFO| LIH|    2447|
|   SFO| MCO|    2445|
|   SFO| IAD|    2419|
|   SFO| HNL|    2398|
+------+----+--------+
only showing top 10 rows



## **Step 7**: Aggregation

In [ ]:
## aggregation
spark.sql("""SELECT concat(Origin, Dest) as market,
avg(Distance)
FROM airlines
GROUP BY 1
ORDER BY 1 DESC""").show(10)

+------+-------------+
|market|avg(Distance)|
+------+-------------+
|SFOTWF|        536.0|
|SFOTUS|        751.0|
|SFOSTL|       1736.0|
|SFOSNA|        372.0|
|SFOSMX|        216.0|
|SFOSMF|         86.0|
|SFOSLC|        599.0|
|SFOSJC|         30.0|
|SFOSEA|        679.0|
|SFOSBP|        191.0|
+------+-------------+
only showing top 10 rows



In [ ]:
## aggregation
spark.sql("""SELECT concat(Origin, Dest) as market,
min(Distance)
FROM airlines
GROUP BY 1
ORDER BY 1 DESC""").show(10)

+------+-------------+
|market|min(Distance)|
+------+-------------+
|SFOTWF|          536|
|SFOTUS|          751|
|SFOSTL|         1736|
|SFOSNA|          372|
|SFOSMX|          216|
|SFOSMF|           86|
|SFOSLC|          599|
|SFOSJC|           30|
|SFOSEA|          679|
|SFOSBP|          191|
+------+-------------+
only showing top 10 rows



In [ ]:
## Top 10
## "SFOSMF" 86 actually is "860" due to the bugs of pyspark show function
spark.sql("""SELECT concat(Origin, Dest) as market,
max(Distance)
FROM airlines
GROUP BY 1
ORDER BY max(Distance) DESC
limit 10""").show(10)

+------+-------------+
|market|max(Distance)|
+------+-------------+
|SFOELP|          993|
|SFODEN|          967|
|SFOCOS|          963|
|SFOBIL|          909|
|SFOABQ|          896|
|SFOSMF|           86|
|SFOASE|          848|
|SFOEGE|          847|
|SFOFCA|          844|
|SFOBZN|          807|
+------+-------------+

